In [1]:
from utils import *
import pandas as pd
import os
import numpy as np

In [2]:
# load data 
file_path = os.path.join("tables", "exerciseTableForBN_python_processed.csv")
exerciseTable = pd.read_csv(file_path)

In [ ]:
plt.scatter(exerciseTable['age'], exerciseTable['gender'])

In [4]:
# Define tiers
tiers = {
    'static': ['age', 'gender', 'diabetesDuration', 'BMI', 'HbA1c', 'InsSensitivity', 'InsCarbRatio'],
    'pre': ['preExerciseRoc', 'startExerciseGlucoseLevel', 'preExerciseGlucoseCV', 'IOBnorm', 'COBnorm', 'AOB', 'TotalCWL'],
    #'exercise': ['MET_min', 'ExerciseModality'],
    'exercise' : ['MET', 'DurationValue', 'ExerciseModality'],
    'outcome_during': ['exerciseMaxExcursion', 'exerciseMaxSpikeRoc', 'exerciseMaxDropRoc', 'exerciseNadir', 'exercisePeak', 'exerciseTIR', 'exerciseTBR', 'exerciseAUC70'],
    'outcome_post': ['maxGlucosePostExercise', 'minGlucosePostExercise','postExerciseTimeToNadir', 'postExerciseTIR','postExerciseTBR','postExerciseTAR', 'postExerciseGlucoseCV', 'postExerciseAUC70']
}

all_features = exerciseTable.columns.tolist()
features_to_drop = set(all_features) - set(sum(tiers.values(), []))

In [10]:
# Discretize data

df = exerciseTable.copy()

df_discrete = discretize_data(df, 
    cv_strategy = "statistical",
    bmi_strategy = "clinical",
    hba1c_strategy = "clinical", 
    glucose_strategy = "clinical", 
    roc_strategy = "clinical",
    cols_to_remove = features_to_drop)

#Check Discretization
check_discretization(df_discrete) # comment if not needed!


# Collapse bins
df_discrete_collapsed = collapse_sparse_bins(df_discrete)

Discretizing features | CV: statistical | BMI: clinical | HbA1c: clinical | Glucose: clinical
   - preExerciseGlucoseCV thresholds: [ 0.          8.78666758 15.89474743 68.63304324]
   - postExerciseGlucoseCV thresholds: [ 1.67142667 15.69501235 24.74186724 64.76435076]
Discretization complete.

--- DISCRETIZATION HEALTH CHECK ---
📊 Feature: DurationValue
      Short: 40.4%
      Medium: 39.6%
      Long: 20.0%
------------------------------
📊 Feature: MET
      Moderate: 50.3%
      Vigorous: 34.9%
      Light: 14.8%
------------------------------
📊 Feature: age
      Younger: 37.8%
      Middle: 36.0%
      Older: 26.3%
------------------------------
📊 Feature: gender
      female: 64.1%
      male: 35.9%
------------------------------
📊 Feature: diabetesDuration
      Short: 36.7%
      Medium: 32.1%
      Long: 31.2%
------------------------------
📊 Feature: HbA1c
      Diabetes: 43.5%
      Prediabetes: 39.2%
      Normal: 17.3%
------------------------------
📊 Feature: AOB
      

/Users/albertogastaldello/Desktop/PAxT1D_BN/LOOP_repo/utils.py:325: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  clean_df['BMI'] = clean_df['BMI'].replace(
/Users/albertogastaldello/Desktop/PAxT1D_BN/LOOP_repo/utils.py:334: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  clean_df['startExerciseGlucoseLevel'] = clean_df['startExerciseGlucoseLevel'].replace(
/Users/albertogastaldello/Desktop/PAxT1D_BN/LOOP_repo/utils.py:340: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will onl

In [12]:
# Visualize Feature Distribution after Discretization (and after bins collapse)

target_feature = 'postExerciseAUC70'

plot_single_feature_distribution(df_discrete, target_feature, title_prefix="- Clinical Thresholds")
plot_single_feature_distribution(df_discrete_collapsed, target_feature, title_prefix="- Clinical Thresholds After Merge")


In [ ]:
# Data Driven BN

collapse = True

if(collapse):
    learner_data_driven = gum.BNLearner(df_discrete_collapsed)
else:
    learner_data_driven = gum.BNLearner(df_discrete)

learner_data_driven.useGreedyHillClimbing()
bn_data_driven = learner_data_driven.learnBN()

visualize_network(bn_data_driven, tiers)

In [ ]:
# Constrained BN

collapse = True

if(collapse):
    learner_constrained = gum.BNLearner(df_discrete_collapsed)
else:
    learner_constrained = gum.BNLearner(df_discrete)
    
learner_constrained.useGreedyHillClimbing()
learner_constrained = apply_expert_constraints(learner_constrained, tiers)
bn_constrained = learner_constrained.learnBN()

visualize_network(bn_constrained, tiers)

INFERENCE

In [ ]:
# Launch the interactive inference GUI
gnb.showInference(bn_constrained, size="15")
# 1. Define the evidence (the nodes you want to "click")
# Make sure these strings exactly match your discretized categories
my_evidence = {
    'ExerciseModality': 'Aerobic', 
}

# 2. Pass the evidence directly into the visualizer!
# pyagrum will calculate the math and draw the updated graph
gnb.showInference(bn_constrained, evs=my_evidence, size="15")


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pyagrum as gum

def interactive_inference_dashboard(bn, evidence_nodes, target_node):
    """
    Generates an interactive ipywidgets dashboard for a pyAgrum Bayesian Network.
    
    Parameters:
    bn (gum.BayesNet): Your trained Bayesian Network.
    evidence_nodes (list of str): The names of the features you want to act as inputs (dropdowns).
    target_node (str): The name of the outcome feature you want to visualize.
    """
    # 1. Initialize the Inference Engine
    ie = gum.LazyPropagation(bn)

    # 2. Extract Target States
    target_states = list(bn.variableFromName(target_node).labels())

    # 3. Dynamically Generate Dropdowns for Evidence Nodes
    dropdowns = {}
    for node in evidence_nodes:
        # Extract states for this specific node directly from the network
        states = list(bn.variableFromName(node).labels())
        
        # Create the widget
        dropdown = widgets.Dropdown(
            options=['Not Fixed'] + states,
            description=f'{node}:',
            style={'description_width': 'initial'}
        )
        dropdowns[node] = dropdown

    # 4. Create the Output Plot Area
    out_plot = widgets.Output()

    # 5. The Dynamic Update Function
    def update_dashboard(*args):
        with out_plot:
            clear_output(wait=True)
            ie.eraseAllEvidence()
            
            # Dynamically inject evidence from all generated dropdowns
            for node_name, dropdown_widget in dropdowns.items():
                if dropdown_widget.value != 'Not Fixed':
                    ie.addEvidence(node_name, dropdown_widget.value)
                    
            # Perform Inference
            ie.makeInference()
            posterior = ie.posterior(target_node)
            
            # Extract probabilities
            probs = [posterior[i] * 100 for i in range(len(target_states))]
            
            # Draw the Chart
            plt.figure(figsize=(8, 4))
            bars = plt.bar(target_states, probs, color='#2ca02c', edgecolor='black')
            plt.ylim(0, 100)
            plt.ylabel('Probability (%)', fontsize=12)
            plt.title(f'Predicted Outcome: {target_node}', fontsize=14, fontweight='bold')
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            
            # Add percentage labels
            for bar in bars:
                yval = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2, yval + 1.5, f'{yval:.1f}%', 
                         ha='center', va='bottom', fontweight='bold')
                
            # Clean layout
            plt.gca().spines['top'].set_visible(False)
            plt.gca().spines['right'].set_visible(False)
            plt.tight_layout()
            plt.show()

    # 6. Bind the Observers
    for dropdown_widget in dropdowns.values():
        dropdown_widget.observe(update_dashboard, names='value')

    # 7. Group the UI Elements dynamically
    # We use HBox for the dropdowns. If there are many, it might be better to wrap them, 
    # but HBox works great for 3-5 variables.
    dropdown_container = widgets.HBox(list(dropdowns.values()))
    
    ui = widgets.VBox([
        widgets.HTML(f"<h3>Dynamic Inference: Impact on <i>{target_node}</i></h3>"),
        dropdown_container,
        out_plot
    ])

    # 8. Display and initialize
    display(ui)
    update_dashboard()




# Pass the network, a list of your new features, and your acute outcome
interactive_inference_dashboard(
    bn = bn_constrained, 
    evidence_nodes = [ 'preExerciseRoc', 'age', 'startExerciseGlucoseLevel'], 
    target_node = 'exerciseGlucoseRoc'
)